# Hidden in Plain Sight: Steganography, Frequency Analysis, and the Mathematics of Secrets

## Introduction: The Art and Science of Hiding Information

What if I told you that an innocent-looking image could contain a secret message? Not just hidden inside it through sophisticated compression algorithms, but encoded as the actual visible pixels themselves? This notebook explores a fascinating intersection of information theory, cryptography, and visual representation, demonstrating how the mathematical structures we use to represent digital images provide enormous capacity for encoding textual information.

Throughout this exploration, we will examine three interconnected concepts:

First, we will investigate **steganography**, the practice of concealing messages within other non-secret data. Unlike cryptography, which scrambles a message to make it unreadable, steganography hides the very existence of the message. The word itself derives from the Greek *steganos* (covered or concealed) and *graphein* (writing). This technique has a rich history, from invisible inks and microdots to modern digital watermarking and data exfiltration.

Second, we will explore **frequency analysis**, one of the oldest and most powerful techniques in cryptanalysis. This method, which exploits the statistical properties of language, was first systematically described by the 9th-century Arab mathematician Al-Kindi in his manuscript "On Deciphering Cryptographic Messages." The technique relies on the observation that letters in any given language appear with characteristic frequencies, and these patterns persist even when the letters are substituted with other symbols or, in our case, colors.

Third, we will work with **matrices and visual data representation**, understanding images not as continuous visual fields but as discrete arrays of numerical values. This perspective, fundamental to computer graphics and image processing, reveals that every digital image is essentially a matrix of numbers, and thus a potential carrier for encoded information.

## The Relationship Between Color Space and Information Capacity

Digital images use the RGB color model, where each pixel is represented by three values corresponding to the intensity of red, green, and blue light. In a standard 24-bit color space, each channel can take any value from 0 to 255, giving us 256 possible values per channel.

The total number of distinct colors available is therefore:

```
256 × 256 × 256 = 16,777,216 unique colors
```

Now consider the ASCII character set, which uses 7 or 8 bits to represent text characters. Standard ASCII defines 128 characters (using 7 bits), while extended ASCII uses 256 characters. Even if we restrict ourselves to uppercase English letters, spaces, and basic punctuation, we need to represent perhaps 30-40 distinct symbols.

The ratio between what we have (millions of colors) and what we need (dozens of characters) reveals an enormous information theoretic surplus. This surplus is what makes color-based text encoding not only possible but remarkably robust. We could assign thousands of unique colors to each letter and still have room left over.

### Historical Context: From Ancient Techniques to Digital Methods

Before we begin our practical work, it's worth considering the historical continuity of these techniques. Steganography is ancient. Herodotus describes Histiaeus tattooing a message on a slave's shaved head and waiting for the hair to grow back before sending him as a messenger. During World War II, the Allies used microdots (photographs reduced to the size of a period) to hide messages in seemingly innocent letters.

Frequency analysis, meanwhile, fundamentally changed the history of cryptography. For centuries, substitution ciphers like the Caesar cipher were considered unbreakable. Al-Kindi's insight that different letters appear with different frequencies meant that even complex substitution schemes could be attacked systematically. This led to an arms race between code makers and code breakers that continues to this day, though now the battleground involves quantum computers and mathematical complexity theory rather than letter frequency tables.

What we will do in this notebook is modest compared to modern cryptographic systems like AES or RSA, but it connects directly to these historical traditions and illuminates the fundamental principles that underlie all information hiding and code breaking.

### Why This Matters: Matrices, Representation, and Computational Thinking

One of the central insights of computer science is that many different kinds of data can be represented as arrays of numbers. Images are not special or separate from other data types; they are simply matrices where each entry happens to be interpreted as a color value by display hardware. Understanding this equivalence between images and numerical arrays is crucial for anyone working with digital media, data visualization, or machine learning.

When you understand that an image is just a two-dimensional array of numbers, you can start to ask interesting questions: Can we perform mathematical operations on images? Can we extract statistical patterns from them? Can we encode other types of information within them? The answer to all these questions is yes, and this notebook provides a concrete demonstration of some of these possibilities.

Furthermore, this exercise demonstrates why abstract mathematical concepts like matrices, arrays, and statistical distributions are not merely academic exercises but practical tools for solving real problems. The connection between theory and practice is not always obvious in mathematics education, but projects like this one make that connection explicit and tangible.

## Part 1: Libraries and Computational Infrastructure

We will be using several Python libraries that provide us with the computational infrastructure for this exploration:

**Matplotlib** is Python's foundational plotting library, created by John Hunter in 2003. It was originally designed to provide MATLAB-like plotting capabilities within Python and has since become the most widely used data visualization library in the Python ecosystem. We will use it primarily for displaying images (which, remember, are just matrices of color values) and for creating statistical visualizations.

**Collections.Counter** is a specialized dictionary subclass that provides an elegant way to count hashable objects. In our case, we will use it to count the frequency of letters in text and the frequency of colors in images. The Counter class demonstrates how Python's built-in data structures can be extended and specialized for particular use cases.

**Random** provides various functions for generating pseudo-random numbers. We will use it to create our cipher by generating random color assignments for each letter. It's worth noting that these are not truly random but rather pseudo-random, generated by deterministic algorithms that produce sequences of numbers that pass statistical tests for randomness. For our purposes, this is more than sufficient, though for cryptographic applications in production systems, the `secrets` module would be more appropriate.

These libraries represent different aspects of the Python ecosystem: Matplotlib comes from the scientific computing community, Counter is part of Python's standard library and reflects Pythonic design principles, and the random module provides essential functionality that every programming environment needs.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import random

# Configure matplotlib for better default aesthetics
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully.")
print("\nPython version information:")
import sys
print(f"Python {sys.version}")

## Part 2: Simple ASCII-Based Encoding

### Understanding ASCII and Character Encoding

Before digital computers, there was no standard way to represent text electronically. Different manufacturers used different encoding schemes, which made data exchange difficult. ASCII (American Standard Code for Information Interchange), developed in the 1960s, provided a standard mapping between characters and numbers.

ASCII uses 7 bits to represent 128 different characters, including:
- The uppercase letters A-Z (values 65-90)
- The lowercase letters a-z (values 97-122)
- Digits 0-9 (values 48-57)
- Various punctuation marks and control characters

In Python, we can convert between characters and their ASCII values using the built-in functions `ord()` (which gives us the ASCII value of a character) and `chr()` (which gives us the character for a given ASCII value).

### A Simple Mapping Strategy

Our first approach will be to create a deterministic mapping from ASCII values to colors. We will use a simple arithmetic transformation: we'll multiply the ASCII value by different prime numbers (3, 7, and 11) and take the result modulo 256 to get values in the range 0-255 for each color channel.

Why prime numbers? Prime numbers help us avoid patterns that might emerge from common factors. For example, if we used 2, 4, and 6, certain relationships between characters might produce similar colors. The use of coprime multipliers (numbers that share no common factors) helps ensure that different characters map to visually distinct colors.

This approach is not cryptographically secure in any meaningful sense. Anyone who knows the formula can decode the message immediately. However, it serves as a useful introduction to the concept of encoding information in color values, and it demonstrates how mathematical transformations can be used to create bijective (one-to-one) mappings between different data representations.

In [ ]:
def text_to_ascii_colors(message):
    """
    Convert text to RGB colors using a deterministic transformation of ASCII values.
    
    This function creates a visual representation of text where each character
    is mapped to a unique color based on its ASCII value. The mapping uses
    prime number multipliers to create visually distinct colors.
    
    Parameters:
    -----------
    message : str
        The text message to encode as colors
    
    Returns:
    --------
    list of lists
        Each inner list contains [r, g, b] values normalized to the range [0, 1]
        as required by matplotlib's imshow function.
    """
    colors = []
    
    for char in message:
        # Get the ASCII value of the character
        ascii_val = ord(char)
        
        # Create RGB values using prime number multipliers
        # The modulo operation ensures values stay in the range 0-255
        red = (ascii_val * 3) % 256
        green = (ascii_val * 7) % 256
        blue = (ascii_val * 11) % 256
        
        # Normalize to [0, 1] range for matplotlib
        colors.append([red/255, green/255, blue/255])
    
    return colors

def create_image_matrix(colors, width=None):
    """
    Arrange a flat list of colors into a 2D matrix for display as an image.
    
    This function demonstrates the fundamental relationship between linear
    sequences of data and two-dimensional matrix representations. The choice
    of width determines the aspect ratio of the resulting image.
    
    Parameters:
    -----------
    colors : list of lists
        List of RGB color values
    width : int, optional
        Number of pixels per row. If None, creates approximately square image.
    
    Returns:
    --------
    list of lists
        A 2D array (list of rows) where each element is an RGB color triplet
    """
    num_pixels = len(colors)
    
    # If width not specified, aim for a roughly square layout
    if width is None:
        width = int(num_pixels ** 0.5) + 1
    
    # Calculate how many rows we need
    height = (num_pixels // width) + (1 if num_pixels % width != 0 else 0)
    
    # Build the 2D matrix
    image = []
    for row in range(height):
        image_row = []
        for col in range(width):
            idx = row * width + col
            if idx < num_pixels:
                image_row.append(colors[idx])
            else:
                # Fill remaining space with black
                image_row.append([0, 0, 0])
        image.append(image_row)
    
    return image

# Test our encoding with a simple message
test_message = "HELLO WORLD"
print(f"Encoding the message: '{test_message}'")
print(f"Message length: {len(test_message)} characters\n")

# Show the ASCII values for the first few characters
print("ASCII values of first characters:")
for i in range(min(5, len(test_message))):
    char = test_message[i]
    print(f"  '{char}' -> {ord(char)}")

### Visualizing the Encoded Message

Now we will create a visual representation of our encoded message. Each character becomes a single pixel in our image, with its color determined by the mathematical transformation we defined above.

When you look at the result, you're seeing a direct visual mapping of ASCII values through a simple arithmetic transformation. The image contains no hidden compression or complex encoding; it is simply the message itself represented in a different modality (color instead of text).

This is a key insight: information can be represented in multiple ways, and sometimes changing the representation reveals new possibilities or makes certain operations easier. This principle underlies much of computer science, from different sorting algorithms to various data compression techniques.

In [ ]:
# Encode our test message
colors = text_to_ascii_colors(test_message)
image = create_image_matrix(colors, width=6)

# Display the encoded image
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(image)
ax.set_title(f"ASCII-Encoded Message ({len(test_message)} characters)", 
             fontsize=14, pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

print("The image above contains our encoded message.")
print("Each colored square represents one character.")
print("The colors are determined by a mathematical transformation of ASCII values.")

### Decoding: Reversing the Transformation

Since our encoding function is deterministic and invertible, we can recover the original message by trying all possible ASCII values and finding which one produces colors matching what we see in the image.

This brute-force approach works well here because:
1. The search space is small (only 128 standard ASCII characters)
2. The mapping is one-to-one (each ASCII value produces a unique color)
3. We know the exact transformation function

In a real steganographic system, the decoding would need to be more sophisticated, perhaps requiring a secret key or knowledge of a hidden algorithm. But for our pedagogical purposes, this simple invertible transformation serves to illustrate the core concepts.

In [ ]:
def colors_to_ascii_text(colors):
    """
    Decode colors back to text by finding matching ASCII values.
    
    This function implements a brute-force search through ASCII character
    space to find which character would produce each observed color.
    
    Parameters:
    -----------
    colors : list of lists
        RGB color values in the range [0, 1]
    
    Returns:
    --------
    str
        The decoded text message
    """
    message = ""
    
    for color in colors:
        # Convert from [0,1] back to [0,255]
        red = int(color[0] * 255)
        green = int(color[1] * 255)
        blue = int(color[2] * 255)
        
        # Try all possible ASCII values
        for ascii_val in range(128):
            # Compute what colors this ASCII value would produce
            test_r = (ascii_val * 3) % 256
            test_g = (ascii_val * 7) % 256
            test_b = (ascii_val * 11) % 256
            
            # If we find a match, we've found our character
            if test_r == red and test_g == green and test_b == blue:
                message += chr(ascii_val)
                break
    
    return message

# Decode our image
decoded = colors_to_ascii_text(colors)

print("\nDecoding Results:")
print("=" * 50)
print(f"Original message: '{test_message}'")
print(f"Decoded message:  '{decoded}'")
print(f"\nSuccessful recovery: {test_message == decoded}")

### Reflection Questions

Before moving on, consider these questions:

1. What are the advantages and disadvantages of this encoding scheme? Could someone looking at the image tell it contains a message?

2. Why did we use prime numbers (3, 7, 11) as multipliers? What would happen if we used numbers with common factors?

3. This encoding is completely deterministic (no randomness or secret keys). What are the security implications of this choice?

4. How would you modify the encoding to make it less obvious that the image contains hidden information? What trade-offs would you have to make?

### Extension Exercise: Encoding Your Own Messages

Now that you understand the basic mechanism, try encoding your own messages. Consider trying messages of different lengths and with different character types (uppercase, lowercase, numbers, punctuation). Observe how the resulting images differ.

You might also experiment with the width parameter when creating the image matrix. How does the aspect ratio affect the visual appearance? Are there patterns that emerge when messages are arranged in particular ways?

In [ ]:
# Try your own message here
my_message = "INSERT YOUR MESSAGE HERE"

# Encode and display
my_colors = text_to_ascii_colors(my_message)
my_image = create_image_matrix(my_colors, width=8)

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(my_image)
ax.set_title("Your Encoded Message", fontsize=14, pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

# Verify we can decode it
recovered = colors_to_ascii_text(my_colors)
print(f"\nOriginal: {my_message}")
print(f"Decoded:  {recovered}")
print(f"Match: {my_message == recovered}")

## Part 3: Frequency Analysis and the Statistical Properties of Language

### The Problem with Simple Substitution

Our ASCII encoding above has a fundamental weakness: it's completely deterministic. If you know the encoding function, you can immediately decode any message. But what if we scrambled the mapping? What if 'A' mapped to a random color, 'B' to another random color, and so on?

This would seem to make decoding much harder. Without knowing the secret mapping (the cipher), you can't simply reverse a formula to recover the message. This is essentially a substitution cipher, one of the oldest forms of encryption. The Caesar cipher, which shifts each letter by a fixed number of positions in the alphabet, is a special case of this.

However, substitution ciphers have a critical vulnerability that was discovered over a thousand years ago: they preserve the frequency patterns of the underlying language. If 'E' is the most common letter in English and we substitute it with 'X', then 'X' will be the most common letter in our encrypted text. This observation provides a powerful attack vector.

### The History of Frequency Analysis

The technique of frequency analysis was first systematically described by the 9th-century Arab mathematician Abu Yusuf Ya'qub ibn Ishaq al-Kindi, known as Al-Kindi. His manuscript "On Deciphering Cryptographic Messages" (Risalah fi Istikhraj al-Mu'amma) outlined a methodical approach to breaking substitution ciphers by analyzing the frequency of letters.

Al-Kindi's work built on earlier observations about language patterns, but he was the first to formalize this into a cryptanalytic technique. His insight fundamentally changed cryptography, rendering simple substitution ciphers obsolete as secure encryption methods (though they continued to be used by those unaware of frequency analysis).

This historical development illustrates an important pattern in the history of cryptography: advances in mathematics and statistical analysis have repeatedly broken supposedly secure encryption schemes, leading to an ongoing arms race between cipher makers and cipher breakers.

### Letter Frequencies in English

In English text, letters appear with markedly different frequencies. The letter 'E' appears in approximately 12.7% of all letters in typical English text, while 'Z' appears in only about 0.07% of letters. This massive difference provides statistical leverage for breaking substitution ciphers.

The standard ordering of letter frequency in English (from most to least common) is often remembered by the mnemonic "ETAOIN SHRDLU," which represents the approximate order. However, these frequencies can vary depending on the type of text (technical documents have different letter distributions than novels, for example).

We will use a sample of English text to empirically determine letter frequencies, then use those frequencies to crack an encrypted message.

### Analyzing Letter Frequencies in English Text

To perform frequency analysis, we first need a representative sample of English text. We'll use an excerpt from Shakespeare, which provides reasonably representative letter frequencies while also being a public domain text that we can freely use.

The choice of training text matters. Different types of writing have slightly different letter distributions. Scientific papers use more letters like 'X' and 'Z' (for variables and technical terms), while casual conversation uses more common words with high-frequency letters. For general-purpose frequency analysis, classic literature like Shakespeare provides a good balance.

We're also intentionally using a fairly long text sample. Statistical analysis becomes more accurate with larger sample sizes, a principle that applies across many domains of data science and scientific research.

In [ ]:
# A substantial English text sample for frequency analysis
# This combines several well-known Shakespeare passages
training_text = """
To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them To die to sleep
No more and by a sleep to say we end
The heart ache and the thousand natural shocks
That flesh is heir to tis a consummation
Devoutly to be wished To die to sleep
To sleep perchance to dream ay theres the rub
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil
Must give us pause theres the respect
That makes calamity of so long life
For who would bear the whips and scorns of time
The oppressor's wrong the proud man's contumely
The pangs of despised love the law's delay
The insolence of office and the spurns
That patient merit of the unworthy takes
When he himself might his quietus make
With a bare bodkin Who would fardels bear
To grunt and sweat under a weary life
But that the dread of something after death
The undiscovered country from whose bourn
No traveller returns puzzles the will
And makes us rather bear those ills we have
Than fly to others that we know not of
Thus conscience does make cowards of us all
And thus the native hue of resolution
Is sicklied oer with the pale cast of thought
And enterprises of great pith and moment
With this regard their currents turn awry
And lose the name of action
All the world is a stage and all the men and women merely players
They have their exits and their entrances And one man in his time plays many parts
His acts being seven ages At first the infant mewling and puking in the nurses arms
Then the whining schoolboy with his satchel and shining morning face
creeping like snail unwillingly to school And then the lover sighing like furnace
with a woeful ballad made to his mistress eyebrow Then a soldier full of strange oaths
and bearded like the pard jealous in honor sudden and quick in quarrel
seeking the bubble reputation even in the cannons mouth And then the justice
in fair round belly with good capon lined with eyes severe and beard of formal cut
full of wise saws and modern instances and so he plays his part
The sixth age shifts into the lean and slippered pantaloon with spectacles on nose
and pouch on side his youthful hose well saved a world too wide for his shrunk shank
and his big manly voice turning again toward childish treble pipes and whistles in his sound
Last scene of all that ends this strange eventful history is second childishness
and mere oblivion sans teeth sans eyes sans taste sans everything
""" * 3  # Repeat to increase sample size

print(f"Training text length: {len(training_text)} characters")
print(f"Estimated word count: {len(training_text.split())} words")

### Computing Frequency Distributions

We will now analyze our text sample to determine how often each letter appears. This process involves several steps:

1. **Normalization**: Convert all text to uppercase to treat 'A' and 'a' as the same letter
2. **Filtering**: Keep only letters and spaces, discarding punctuation
3. **Counting**: Use Python's Counter to tally occurrences
4. **Sorting**: Arrange characters by frequency (most common first)

The Counter class from Python's collections module provides an elegant solution for this type of problem. It automatically handles the bookkeeping of counting occurrences and provides methods for finding the most common elements.

In [ ]:
def analyze_letter_frequencies(text):
    """
    Analyze the frequency of letters and spaces in a text sample.
    
    This function performs the text normalization and counting necessary
    for frequency analysis. It preserves spaces as distinct symbols since
    word boundaries are often helpful in cryptanalysis.
    
    Parameters:
    -----------
    text : str
        The text to analyze
    
    Returns:
    --------
    list of tuples
        Each tuple is (character, count), sorted by frequency (descending)
    """
    # Normalize: uppercase and filter
    clean_text = ""
    for char in text.upper():
        if char.isalpha() or char == ' ':
            clean_text += char
    
    # Count occurrences
    letter_counts = Counter(clean_text)
    
    # Sort by frequency (most common first)
    return letter_counts.most_common()

# Analyze our training text
english_frequencies = analyze_letter_frequencies(training_text)

# Display results
print("Letter Frequencies in English (from our sample)\n")
print("="*60)
print(f"{'Character':<12} {'Count':<10} {'Percentage':<12}")
print("="*60)

total_chars = sum(count for _, count in english_frequencies)

for char, count in english_frequencies[:15]:  # Show top 15
    percentage = (count / total_chars) * 100
    char_display = f"'{char}'"
    print(f"{char_display:<12} {count:<10} {percentage:>6.2f}%")

print("\n" + "="*60)
print(f"Total characters analyzed: {total_chars}")

### Visualizing Letter Frequencies

Statistical data is often more intuitive when visualized. We'll create a bar chart showing the frequency of each letter. This type of visualization immediately reveals the enormous variation in letter usage that makes frequency analysis possible.

Notice how the space character (representing word boundaries) is typically the most common "character" in any text. After spaces, common letters like E, T, A, and O dominate, while rare letters like Z, Q, and X barely register.

This distribution is not uniform or random; it reflects the structure and history of the English language, including the words we borrowed from other languages, the way we form plurals and past tenses, and the phonetic patterns that make certain letter combinations more common than others.

In [ ]:
# Prepare data for visualization
top_n = 20
characters = [char for char, _ in english_frequencies[:top_n]]
counts = [count for _, count in english_frequencies[:top_n]]

# Create bar chart
fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.bar(range(len(characters)), counts, color='steelblue', 
              edgecolor='navy', linewidth=1.2)

# Highlight the most common character
bars[0].set_color('crimson')

# Configure axes
ax.set_xlabel('Character', fontsize=13, fontweight='bold')
ax.set_ylabel('Frequency (count)', fontsize=13, fontweight='bold')
ax.set_title('Letter Frequency Distribution in English Text Sample', 
             fontsize=15, fontweight='bold', pad=20)
ax.set_xticks(range(len(characters)))
ax.set_xticklabels([f"'{c}'" if c != ' ' else "' '" for c in characters])
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f"\nMost common character: '{characters[0]}' with {counts[0]} occurrences")
print("This frequency distribution is characteristic of English text.")
print("Different languages have different frequency patterns.")

### Reflection: Language Patterns and Statistical Analysis

The frequency distribution you see above is not random. It emerges from:

**Historical factors**: English has borrowed heavily from Latin, French, and Germanic languages, each with their own phonetic patterns. This history is reflected in our letter frequencies.

**Structural features**: English uses the letter 'E' in many grammatical constructions (past tense '-ed', plural '-es', and superlative '-est'). The letter is also common in the most frequent English words like 'the', 'be', 'we', 'he', 'she', etc.

**Phonetic constraints**: Certain sound combinations are easier to pronounce than others, which affects which letters appear together and how often they appear overall.

These patterns are what make frequency analysis possible. They're also what make it difficult to create truly random-looking encrypted text using simple substitution. The structure of language shows through the encryption.

## Part 4: Creating and Breaking a Substitution Cipher

### Designing a Random Substitution Cipher

Now we will create a more sophisticated encoding scheme. Instead of using a deterministic mathematical formula to map characters to colors, we will generate a random mapping. Each letter and space will be assigned a unique, randomly chosen color.

This approach has several advantages over our ASCII encoding:
1. Without knowing the random mapping (the key), you cannot easily decode the message
2. The same message will look different each time you encode it (if you use a different random seed)
3. There's no obvious mathematical pattern in the color choices

However, as we will see, this cipher still has a fatal flaw: it preserves the frequency distribution of the underlying text. The most common letter in English will be encoded with the most common color in our image, even though we don't know in advance which color that will be.

In [ ]:
def create_random_cipher(seed=None):
    """
    Generate a random substitution cipher mapping characters to colors.
    
    This function creates a one-to-one mapping between characters and RGB colors.
    The mapping is random but deterministic (given the same seed, it produces
    the same cipher).
    
    Parameters:
    -----------
    seed : int, optional
        Random seed for reproducibility. If None, truly random mapping.
    
    Returns:
    --------
    dict
        Mapping from characters to RGB tuples (values in range 0-255)
    """
    if seed is not None:
        random.seed(seed)
    
    cipher = {}
    used_colors = set()
    
    # All characters we want to encode
    characters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ '
    
    for char in characters:
        # Generate random, unique colors
        while True:
            color = (random.randint(0, 255), 
                    random.randint(0, 255), 
                    random.randint(0, 255))
            
            # Ensure each character gets a unique color
            if color not in used_colors:
                used_colors.add(color)
                break
        
        cipher[char] = color
    
    return cipher

def encode_with_substitution(message, cipher):
    """
    Encode a message using a substitution cipher.
    
    Parameters:
    -----------
    message : str
        Text to encode
    cipher : dict
        Mapping from characters to RGB tuples
    
    Returns:
    --------
    list of lists
        RGB values normalized to [0, 1] range
    """
    colors = []
    for char in message.upper():
        if char in cipher:
            rgb = cipher[char]
            # Normalize to [0, 1]
            colors.append([rgb[0]/255, rgb[1]/255, rgb[2]/255])
    return colors

# Create our secret cipher
secret_cipher = create_random_cipher(seed=42)  # Seed for reproducibility

# Create a longer message for more statistical significance
secret_message = "THE QUICK BROWN FOX JUMPS OVER THE LAZY DOG"

print("Substitution Cipher Created")
print("="*60)
print(f"Message to encode: '{secret_message}'")
print(f"Message length: {len(secret_message)} characters\n")

# Show a few example mappings
print("Sample character-to-color mappings:")
for char in ['E', 'T', 'A', ' ']:
    rgb = secret_cipher[char]
    print(f"  '{char}' -> RGB{rgb}")

print("\nEach character has a unique, randomly assigned color.")

### Visualizing the Encrypted Message

Now we'll encode our message using the random cipher and display it as an image. To someone without knowledge of the cipher, this looks like a collection of random colors. There's no obvious pattern or structure that would suggest it contains a message.

However, if you were to carefully count which colors appear most frequently, you would discover that some colors appear much more often than others. This frequency distribution is not random; it reflects the frequency distribution of letters in the underlying English text.

This is the weakness we will exploit to crack the cipher.

In [ ]:
# Encode the message
encrypted_colors = encode_with_substitution(secret_message, secret_cipher)
encrypted_image = create_image_matrix(encrypted_colors, width=10)

# Display the encrypted image
fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(encrypted_image)
ax.set_title("Encrypted Message Using Substitution Cipher", 
             fontsize=15, fontweight='bold', pad=20)
ax.axis('off')
plt.tight_layout()
plt.show()

print("The image above contains an encrypted message.")
print("Each color represents a character, but the mapping is random.")
print("Without the cipher key, direct decoding is not possible.")
print("\nHowever, frequency analysis can still break this encryption...")

## Part 5: Cryptanalysis Through Frequency Analysis

### The Attack Strategy

Our approach to breaking the substitution cipher relies on a simple but powerful observation:

**The most common color in the encrypted image probably represents the most common letter in English.**

More specifically, our attack proceeds as follows:

1. Count how many times each color appears in the encrypted image
2. Sort the colors by frequency (most common first)
3. Create a mapping where:
   - The most common color maps to the most common English letter (probably space or 'E')
   - The second most common color maps to the second most common letter
   - And so on...
4. Use this derived mapping to decrypt the message

This attack works because substitution preserves statistical structure. Even though we've scrambled which color represents which letter, we haven't changed how often each letter appears. The linguistic patterns persist through the encryption.

### Why This Attack Works

The success of frequency analysis demonstrates a fundamental principle in cryptography: **encryption should not preserve statistical properties of the plaintext**. Modern encryption schemes like AES go to great lengths to ensure that the encrypted text has uniform statistical properties, making frequency analysis impossible.

The historical importance of this cannot be overstated. For centuries, people believed substitution ciphers were secure. The discovery of frequency analysis completely changed cryptography, forcing the development of more sophisticated techniques like polyalphabetic ciphers (where the substitution mapping changes for each character).

In [ ]:
def analyze_color_frequencies(colors):
    """
    Count the frequency of each color in an encoded image.
    
    Parameters:
    -----------
    colors : list of lists
        RGB color values in [0, 1] range
    
    Returns:
    --------
    list of tuples
        Each tuple is (RGB tuple, count), sorted by frequency
    """
    # Convert colors to tuples for hashing
    color_tuples = []
    for color in colors:
        r = int(color[0] * 255)
        g = int(color[1] * 255)
        b = int(color[2] * 255)
        color_tuples.append((r, g, b))
    
    # Count and sort by frequency
    return Counter(color_tuples).most_common()

# Analyze the encrypted image
color_frequencies = analyze_color_frequencies(encrypted_colors)

# Display results
print("Color Frequency Analysis")
print("="*60)
print(f"{'Rank':<6} {'RGB Color':<20} {'Count':<8} {'Percentage':<12}")
print("="*60)

total_colors = len(encrypted_colors)

for rank, (color, count) in enumerate(color_frequencies[:10], 1):
    percentage = (count / total_colors) * 100
    print(f"{rank:<6} {str(color):<20} {count:<8} {percentage:>6.2f}%")

print("\n" + "="*60)
print(f"Total pixels analyzed: {total_colors}")
print(f"Unique colors found: {len(color_frequencies)}")

### Comparing Frequency Distributions

Let's visualize both our English letter frequencies and our color frequencies side by side. You should notice a similar pattern in both: a few very common items, a larger group of moderately common items, and a long tail of rare items.

This similarity is not coincidental. The color frequency distribution is a direct reflection of the letter frequency distribution, just with the mapping scrambled. Our task is to unscramble that mapping by aligning the two distributions.

In [ ]:
# Create side-by-side frequency comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Letter frequencies
n_show = 15
letters = [char for char, _ in english_frequencies[:n_show]]
letter_counts = [count for _, count in english_frequencies[:n_show]]

ax1.bar(range(len(letters)), letter_counts, color='steelblue', 
        edgecolor='navy', linewidth=1.2)
ax1.set_xlabel('Character', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('English Letter Frequencies\n(from training text)', 
              fontsize=13, fontweight='bold')
ax1.set_xticks(range(len(letters)))
ax1.set_xticklabels([f"'{c}'" if c != ' ' else "' '" for c in letters], 
                     rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Color frequencies  
color_counts_top = [count for _, count in color_frequencies[:n_show]]

ax2.bar(range(len(color_counts_top)), color_counts_top, color='coral', 
        edgecolor='darkred', linewidth=1.2)
ax2.set_xlabel('Color (by rank)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax2.set_title('Color Frequencies\n(in encrypted image)', 
              fontsize=13, fontweight='bold')
ax2.set_xticks(range(len(color_counts_top)))
ax2.set_xticklabels([f"{i+1}" for i in range(len(color_counts_top))])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice the similar pattern in both distributions:")
print("- A few very common items (letters/colors)")
print("- A gradual decline in frequency")
print("- This similarity is what makes frequency analysis possible.")

### Performing the Attack: Creating a Decryption Mapping

Now we'll implement the actual attack. We'll match the most common color to the most common letter, the second most common color to the second most common letter, and so on.

This is a straightforward attack, but it demonstrates the power of statistical analysis in cryptography. By exploiting the statistical properties of language, we can break an encryption scheme that would seem secure at first glance.

In [ ]:
def crack_substitution_cipher(encrypted_colors, language_frequencies):
    """
    Use frequency analysis to break a substitution cipher.
    
    This function matches color frequencies to letter frequencies,
    assuming that the most common color represents the most common letter.
    
    Parameters:
    -----------
    encrypted_colors : list of lists
        The encrypted message as RGB values
    language_frequencies : list of tuples
        Letter frequency data from training text
    
    Returns:
    --------
    dict
        Decryption mapping from RGB tuples to characters
    """
    # Get color frequencies from encrypted message
    color_freq = analyze_color_frequencies(encrypted_colors)
    
    # Create decryption mapping by matching frequencies
    decryption_map = {}
    
    for i, (color, _) in enumerate(color_freq):
        if i < len(language_frequencies):
            # Most common color gets most common letter, etc.
            letter = language_frequencies[i][0]
            decryption_map[color] = letter
    
    return decryption_map

def decrypt_message(encrypted_colors, decryption_map):
    """
    Apply a decryption mapping to recover the original message.
    
    Parameters:
    -----------
    encrypted_colors : list of lists
        RGB values representing the encrypted message
    decryption_map : dict
        Mapping from RGB tuples to characters
    
    Returns:
    --------
    str
        The decrypted text
    """
    message = ""
    
    for color in encrypted_colors:
        # Convert color back to integer RGB
        r = int(color[0] * 255)
        g = int(color[1] * 255)
        b = int(color[2] * 255)
        rgb = (r, g, b)
        
        # Look up the corresponding letter
        if rgb in decryption_map:
            message += decryption_map[rgb]
        else:
            message += '?'  # Unknown mapping
    
    return message

print("Attempting to crack the cipher using frequency analysis...")
print("="*70)

# Perform the attack
cracked_mapping = crack_substitution_cipher(encrypted_colors, english_frequencies)
decrypted_text = decrypt_message(encrypted_colors, cracked_mapping)

# Display results
print("\nDECRYPTION RESULTS:")
print("="*70)
print(f"Original message:  {secret_message}")
print(f"Decrypted message: {decrypted_text}")
print("="*70)

# Calculate accuracy
if len(secret_message) == len(decrypted_text):
    correct = sum(1 for i in range(len(secret_message)) 
                 if secret_message[i] == decrypted_text[i])
    accuracy = (correct / len(secret_message)) * 100
    
    print(f"\nAccuracy: {accuracy:.1f}%")
    print(f"Characters correct: {correct} out of {len(secret_message)}")
    
    if accuracy == 100:
        print("\nPerfect decryption! Frequency analysis has completely broken the cipher.")
    elif accuracy > 85:
        print("\nExcellent result! Minor errors are normal with shorter messages.")
    elif accuracy > 70:
        print("\nGood result. With a longer message, accuracy would improve.")
    else:
        print("\nPartial success. Longer training text might improve results.")
else:
    print("\nLength mismatch - decryption incomplete.")

### Understanding the Results

The success or failure of our frequency analysis depends on several factors:

**Message length**: Longer messages provide more statistical data, making the frequency patterns clearer. With very short messages, frequency analysis may produce incorrect guesses because the sample size is too small to reliably represent typical English frequencies.

**Training text quality**: Our English frequency data comes from Shakespeare. If the encrypted message uses significantly different vocabulary (technical jargon, for example), the frequency patterns might not match well.

**Character diversity**: Messages that use only a few different letters provide less statistical leverage than messages that exercise more of the alphabet.

In practice, historical cryptanalysts working with substitution ciphers would use additional techniques beyond simple frequency analysis:
- Looking for common short words ("the", "and", "of")
- Analyzing letter pairs (digrams) and triplets (trigrams)
- Using contextual knowledge about the likely content
- Making educated guesses and checking if they lead to readable text

Our automated approach demonstrates the core principle, but human cryptanalysts historically employed a combination of statistical analysis and linguistic intuition.

### Examining the Decryption Mapping

Let's look at some specific examples of how colors were mapped to letters during our decryption attempt. This helps us understand what the algorithm did and why it might have made certain mistakes.

In [ ]:
print("Sample Decryption Mappings")
print("="*70)
print(f"{'Rank':<6} {'RGB Color':<22} {'Mapped To':<12} {'Frequency'}")
print("="*70)

color_freq_list = analyze_color_frequencies(encrypted_colors)

for i, (color, count) in enumerate(color_freq_list[:12], 1):
    if color in cracked_mapping:
        letter = cracked_mapping[color]
        letter_display = f"'{letter}'" if letter != ' ' else "' ' (space)"
        print(f"{i:<6} {str(color):<22} {letter_display:<12} {count}")

print("\n" + "="*70)
print("The algorithm matched colors to letters based solely on frequency.")
print("Most common color -> most common letter, and so on.")

## Part 6: Exploration and Extensions

### Experiment with Different Messages

Now that you understand the technique, try encoding and cracking your own messages. Consider these questions:

1. How does message length affect the accuracy of frequency analysis?
2. What happens if you encode a message in a different language?
3. Can you think of ways to make the cipher more resistant to frequency analysis?
4. How would the analysis change if we included lowercase letters and more punctuation?

In [ ]:
# Try encoding and decrypting your own message
your_message = "WRITE YOUR SECRET MESSAGE HERE"

# Create a new random cipher
your_cipher = create_random_cipher(seed=None)  # No seed = truly random

# Encode
your_colors = encode_with_substitution(your_message, your_cipher)
your_image = create_image_matrix(your_colors, width=10)

# Display
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(your_image)
ax.set_title("Your Encrypted Message", fontsize=14, fontweight='bold')
ax.axis('off')
plt.show()

# Attempt to crack it
your_decryption = crack_substitution_cipher(your_colors, english_frequencies)
your_decrypted = decrypt_message(your_colors, your_decryption)

print(f"\nOriginal:  {your_message}")
print(f"Decrypted: {your_decrypted}")

if len(your_message) == len(your_decrypted):
    matches = sum(1 for i in range(len(your_message)) 
                 if your_message[i] == your_decrypted[i])
    print(f"Accuracy: {(matches/len(your_message))*100:.1f}%")

### Advanced Challenge: Improving the Attack

Our frequency analysis is relatively simple: it just matches the most common colors to the most common letters. You might improve this by:

**Bigram analysis**: Instead of looking at single letters, analyze pairs of letters. In English, certain letter pairs (like 'TH', 'HE', 'AN') are much more common than others.

**Word pattern matching**: If you can identify word boundaries (spaces), you can look for common patterns. For example, three-letter words that start and end with the same letter are likely 'THE' or 'ERE'.

**Dictionary checking**: After making a guess at the decryption, check if the resulting words appear in an English dictionary. This can help you refine your guesses.

These techniques represent the evolution of cryptanalysis from simple statistical analysis to more sophisticated linguistic analysis.

## Part 7: Theoretical Considerations and Modern Cryptography

### Why Modern Ciphers Resist Frequency Analysis

Modern encryption systems like AES (Advanced Encryption Standard) are designed to eliminate all statistical patterns. They achieve this through several techniques:

**Diffusion**: Each bit of the ciphertext depends on multiple bits of the plaintext and the key. A small change in the plaintext causes a large, seemingly random change in the ciphertext (this is called the avalanche effect).

**Confusion**: The relationship between the ciphertext and the key should be as complex as possible. Even if you know some plaintext-ciphertext pairs, it should be extremely difficult to determine the key.

**Block operations**: Modern ciphers operate on blocks of data rather than individual characters, which naturally destroys letter frequency patterns.

The concepts we've explored in this notebook (substitution, frequency analysis, statistical patterns in language) laid the groundwork for modern cryptography, even though modern systems have moved far beyond these simple techniques.

### Information Theory and Steganography

Our color-based encoding demonstrates a fundamental principle from information theory: channels with high capacity can encode multiple types of information simultaneously.

The 24-bit color space provides far more information capacity than needed to distinguish between the colors humans can perceive. This "extra" capacity can be used for other purposes, like embedding hidden messages. This principle underlies many steganographic techniques:

**LSB steganography**: Hiding data in the least significant bits of image pixels
**Frequency domain hiding**: Embedding data in the frequency components of images or audio
**Spread spectrum techniques**: Distributing hidden data across many samples

These techniques are used in digital watermarking, covert communication, and data integrity verification.

### The Arms Race: Cryptography vs. Cryptanalysis

The history of cryptography is characterized by an ongoing competition between those who create codes and those who break them. Each advance in cryptanalysis has spurred new developments in cryptography:

- Al-Kindi's frequency analysis broke simple substitution ciphers
- This led to polyalphabetic ciphers (like Vigenère) that resisted frequency analysis
- The Enigma machine was developed for mechanical polyalphabetic encryption
- Alan Turing and others developed electromechanical methods to break Enigma
- Modern computers enabled both very strong encryption (RSA, AES) and new attacks
- Quantum computers threaten current public-key cryptography
- Post-quantum cryptography is being developed in response

This pattern continues today. The techniques we've explored here are ancient by computational standards, but they illustrate enduring principles about information, patterns, and analysis.

## Part 8: Reflection and Broader Connections

### What We've Learned

Through this exploration, we've discovered several important ideas:

**Digital images are matrices**: Understanding this equivalence opens up possibilities for image processing, computer vision, and creative applications like steganography.

**Language has statistical structure**: The non-uniform distribution of letters in natural language is both a vulnerability (for simple encryption) and a tool (for compression and analysis).

**Information can be represented in multiple ways**: The same message can be text, colors, numbers, or many other forms. The representation you choose affects what operations are easy or hard.

**Statistical analysis is powerful**: By counting and comparing frequencies, we can extract information even from encrypted data.

### Connections to Other Domains

The techniques we've explored connect to many other areas:

**Data compression**: File formats like JPEG and MP3 exploit the statistical properties of images and audio, storing frequently occurring patterns more efficiently.

**Natural language processing**: Understanding letter and word frequencies is fundamental to spell checkers, text predictors, and machine translation systems.

**Machine learning**: Many machine learning algorithms look for statistical patterns in data, similar to how frequency analysis finds patterns in encrypted text.

**Bioinformatics**: DNA sequences can be analyzed using similar frequency-based techniques to identify genes and regulatory elements.

**Digital forensics**: Steganographic techniques are used both to hide evidence and to detect hidden information.

### Critical Questions

As we develop and use these technologies, we should also consider their implications:

**Privacy vs. security**: Strong encryption protects privacy but can also shield criminal activity. How do we balance these concerns?

**Surveillance and steganography**: The ability to hide information in plain sight has both beneficial uses (protecting dissidents) and harmful ones (evading detection). Who decides which uses are legitimate?

**Access to cryptography**: Should encryption tools be freely available to everyone? Or should governments have special access ("backdoors") for law enforcement?

**Historical preservation**: As encryption becomes stronger, how do we ensure that historically important encrypted documents can eventually be read by scholars?

These questions don't have simple answers, but they're important to consider as we develop and deploy cryptographic technologies.

## Resources for Further Exploration

### Books and Papers

**On Cryptography and Its History:**
- Singh, Simon. *The Code Book: The Science of Secrecy from Ancient Egypt to Quantum Cryptography* (1999). An accessible history of cryptography from ancient times to the present.
- Kahn, David. *The Codebreakers* (1967). The comprehensive history of secret communication, from ancient times through the 20th century.
- Al-Kindi. *On Deciphering Cryptographic Messages* (9th century). The original text on frequency analysis, translated in various editions.

**On Information Theory:**
- Shannon, Claude. "A Mathematical Theory of Communication" (1948). The foundational paper of information theory.
- Cover, Thomas M., and Joy A. Thomas. *Elements of Information Theory* (2006). A comprehensive textbook on the subject.

**On Steganography:**
- Cox, Ingemar, et al. *Digital Watermarking and Steganography* (2007). A technical treatment of modern steganographic techniques.
- Wayner, Peter. *Disappearing Cryptography* (2009). Discusses practical applications of information hiding.

### Online Resources

**Interactive Tools and Demonstrations:**
- Cryptography and Network Security Resources: https://www.crypto-it.net/
- Practical Cryptography: http://practicalcryptography.com/
- CrypTool: https://www.cryptool.org/ (Educational software for cryptography)

**Academic Resources:**
- Stanford Cryptography Course: https://crypto.stanford.edu/~dabo/courses/OnlineCrypto/
- Coursera Cryptography Courses: Various offerings from different universities
- MIT OpenCourseWare: Mathematics with Computer Science lectures

### Related Topics to Explore

If this notebook interested you, you might also want to investigate:

- **Public-key cryptography**: RSA, elliptic curve cryptography, and the mathematics behind them
- **Hash functions**: SHA-256, MD5, and their uses in data integrity
- **Digital signatures**: How to prove authorship without revealing a secret key
- **Image processing**: Convolution, edge detection, and computer vision
- **Computational linguistics**: Natural language processing and corpus analysis
- **Data compression**: Huffman coding, LZW compression, and entropy

### Python Libraries for Cryptography and Image Processing

If you want to build on what you've learned:

- **cryptography**: Modern cryptographic recipes and primitives
- **PyCrypto/PyCryptodome**: Low-level cryptographic primitives
- **PIL/Pillow**: Image processing in Python
- **OpenCV**: Computer vision and image manipulation
- **NumPy**: Numerical computing with arrays (for image processing)
- **NLTK**: Natural Language Toolkit for text analysis

### Historical Primary Sources

For those interested in the historical development of these ideas:

- Al-Kindi's manuscript on cryptanalysis (9th century, various translations available)
- Leon Battista Alberti's work on polyalphabetic ciphers (15th century)
- Auguste Kerckhoffs' "La Cryptographie Militaire" (1883)
- Claude Shannon's classified work "Communication Theory of Secrecy Systems" (1945, declassified 1949)
- Alan Turing's work on breaking the Enigma cipher (various sources, many recently declassified)

## Conclusion

We've journeyed from simple ASCII encoding through substitution ciphers to frequency analysis and cryptanalysis. Along the way, we've seen how:

- Images are fundamentally matrices of numbers that can encode any kind of information
- Statistical patterns in language persist even through encryption
- Simple counting and comparison can break supposedly secure codes
- The history of cryptography is an ongoing dialogue between code makers and code breakers

The mathematical and computational techniques we've used—matrices, frequency distributions, statistical analysis—are not just academic exercises. They're practical tools that connect to real problems in security, communication, data analysis, and many other domains.

More importantly, this exploration demonstrates how abstract mathematical concepts become concrete and useful when applied to actual problems. The next time you encounter matrices, arrays, or statistical distributions in your studies, you'll know that these aren't just theoretical constructs—they're tools for solving real problems and understanding the world.

The specific techniques we've used here are ancient in computational terms, but the principles they embody—information hiding, statistical analysis, the relationship between representation and manipulation—remain central to modern computer science, data science, and digital security.

Continue exploring, keep questioning, and remember: sometimes the best way to keep a secret is to hide it in plain sight.